In [1]:
import cv2
import os

# Ruta donde se guardarán los rostros
base_path = 'rostros_conocidos'
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Inicializar el clasificador Haar
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Variables para la interfaz
nombre_usuario = ""
escribiendo_nombre = True
guardar_foto = False
contador = 0
face_id = 1

# Coordenadas del botón guardar
boton_coords = (10, 10, 200, 50)

# Función de evento del mouse
def click_event(event, x, y, flags, param):
    global guardar_foto
    x1, y1, x2, y2 = boton_coords
    if event == cv2.EVENT_LBUTTONDOWN and x1 <= x <= x2 and y1 <= y <= y2:
        guardar_foto = True

# Iniciar la cámara
cap = cv2.VideoCapture(0)
cv2.namedWindow("Registrar Empleado")
cv2.setMouseCallback("Registrar Empleado", click_event)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rostros = face_cascade.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=5)

    # Mostrar campo de texto simulado
    cv2.rectangle(frame, (10, 60), (400, 100), (255, 255, 255), -1)
    cv2.putText(frame, f"Nombre: {nombre_usuario}", (15, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

    # Dibujar botón
    x1, y1, x2, y2 = boton_coords
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 128, 255), -1)
    cv2.putText(frame, "Guardar Foto", (x1 + 20, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    for (x, y, w, h) in rostros:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        rostro = gris[y:y+h, x:x+w]
        rostro = cv2.resize(rostro, (150, 150))

        if guardar_foto and nombre_usuario.strip() != "":
            persona_path = os.path.join(base_path, nombre_usuario.strip())
            if not os.path.exists(persona_path):
                os.makedirs(persona_path)

            nombre_archivo = f"{nombre_usuario}_{contador}.jpg"
            cv2.imwrite(os.path.join(persona_path, nombre_archivo), rostro)
            print(f"✅ Imagen guardada en: {persona_path}/{nombre_archivo}")
            contador += 1
            guardar_foto = False

    cv2.imshow("Registrar Empleado", frame)

    key = cv2.waitKey(1) & 0xFF

    if escribiendo_nombre:
        if 32 <= key <= 126:
            nombre_usuario += chr(key)
        elif key == 8 and len(nombre_usuario) > 0:  # Retroceso
            nombre_usuario = nombre_usuario[:-1]
    if key == 13:  # Enter para confirmar nombre
        escribiendo_nombre = False
    if key in [ord('q'), ord('Q')]:
        break

cap.release()
cv2.destroyAllWindows()